# DiaPredict — COMP6002 live demo

In [1]:

import os, copy, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.max_columns", 12)
print("PyTorch:", torch.__version__)


PyTorch: 2.2.2


## 1 — Different datasets, different input features
Two public datasets act as separate simulated clients. Raw records are not combined.

In [2]:
from pathlib import Path

# Main DiaPredict project folder
PROJECT_ROOT = Path.cwd().resolve().parent

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

DATASET1_PATH = DATA_DIR / "diabetes_prediction_dataset.csv"
DATASET2_PATH = DATA_DIR / "diabetes_dataset.csv"

df1 = pd.read_csv(DATASET1_PATH)
df2 = pd.read_csv(DATASET2_PATH)

TARGET1 = "diabetes"
TARGET2 = "diagnosed_diabetes"

# Same Dataset 2 leakage exclusions used in V1.
df2_model = df2.drop(
    columns=[c for c in ["diabetes_stage", "diabetes_risk_score"] if c in df2.columns]
).copy()

X1 = df1.drop(columns=[TARGET1])
y1 = df1[TARGET1].astype(int)

X2 = df2_model.drop(columns=[TARGET2])
y2 = df2_model[TARGET2].astype(int)

print("Dataset 1:", X1.shape)
print("Dataset 2:", X2.shape)

print("\nDataset 1 target distribution:")
print(y1.value_counts(normalize=True).sort_index().round(4))

print("\nDataset 2 target distribution:")
print(y2.value_counts(normalize=True).sort_index().round(4))


Dataset 1: (100000, 8)
Dataset 2: (100000, 28)

Dataset 1 target distribution:
diabetes
0    0.915
1    0.085
Name: proportion, dtype: float64

Dataset 2 target distribution:
diagnosed_diabetes
0    0.4
1    0.6
Name: proportion, dtype: float64


In [3]:
print("Client A only (examples):", sorted(set(X1.columns) - set(X2.columns))[:8])
print("Client B only (examples):", sorted(set(X2.columns) - set(X1.columns))[:8])
print("Shared feature names:", sorted(set(X1.columns) & set(X2.columns)))

Client A only (examples): ['HbA1c_level', 'blood_glucose_level', 'heart_disease', 'hypertension', 'smoking_history']
Client B only (examples): ['alcohol_consumption_per_week', 'cardiovascular_history', 'cholesterol_total', 'diastolic_bp', 'diet_score', 'education_level', 'employment_status', 'ethnicity']
Shared feature names: ['age', 'bmi', 'gender']


## 2 — Prepare a fast, representative demonstration
Stratified sampling preserves the approximate class proportions. This subset is **only** used in the live demo; saved research results below come from the full experiments.

In [4]:

from sklearn.model_selection import train_test_split

# Small stratified samples for a fast live demonstration
X1_demo, _, y1_demo, _ = train_test_split(
    X1, y1, train_size=3000, random_state=SEED, stratify=y1
)
X2_demo, _, y2_demo, _ = train_test_split(
    X2, y2, train_size=3000, random_state=SEED, stratify=y2
)

# First separate the held-out test sets
X1_temp, X1_te, y1_temp, y1_te = train_test_split(
    X1_demo, y1_demo, test_size=0.2, random_state=SEED,
    stratify=y1_demo
)
X2_temp, X2_te, y2_temp, y2_te = train_test_split(
    X2_demo, y2_demo, test_size=0.2, random_state=SEED,
    stratify=y2_demo
)

# Then separate validation data from training data
X1_tr, X1_val, y1_tr, y1_val = train_test_split(
    X1_temp, y1_temp, test_size=0.2, random_state=SEED,
    stratify=y1_temp
)
X2_tr, X2_val, y2_tr, y2_val = train_test_split(
    X2_temp, y2_temp, test_size=0.2, random_state=SEED,
    stratify=y2_temp
)

print("Client A train/val/test:", len(X1_tr), len(X1_val), len(X1_te))
print("Client B train/val/test:", len(X2_tr), len(X2_val), len(X2_te))


Client A train/val/test: 1920 480 600
Client B train/val/test: 1920 480 600


In [5]:
def build_nn_preprocessor(X):
    categorical = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical = [c for c in X.columns if c not in categorical]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    return ColumnTransformer([
        ("num", numeric_pipe, numerical),
        ("cat", categorical_pipe, categorical)
    ])


prep1 = build_nn_preprocessor(X1_tr)
prep2 = build_nn_preprocessor(X2_tr)

A_train = prep1.fit_transform(X1_tr).astype(np.float32)
A_val = prep1.transform(X1_val).astype(np.float32)
A_test = prep1.transform(X1_te).astype(np.float32)

B_train = prep2.fit_transform(X2_tr).astype(np.float32)
B_val = prep2.transform(X2_val).astype(np.float32)
B_test = prep2.transform(X2_te).astype(np.float32)

print("Client A input dimension:", A_train.shape[1])
print("Client B input dimension:", B_train.shape[1])


Client A input dimension: 14
Client B input dimension: 46


## 3 — Client-specific encoders and compatible prediction heads
Each encoder maps its own input space to an eight-dimensional latent representation. Only prediction-head parameters are averaged.

In [6]:
LATENT_DIM = 8

class ClientEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

class SharedPredictor(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, z):
        return self.net(z).squeeze(1)

class ClientModel(nn.Module):
    def __init__(self, encoder, predictor):
        super().__init__()
        self.encoder = encoder
        self.predictor = predictor

    def forward(self, x):
        return self.predictor(self.encoder(x))


In [7]:
def make_loader(X, y, batch_size=512, shuffle=True):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(np.asarray(y), dtype=torch.float32)
    return DataLoader(
        TensorDataset(X_t, y_t),
        batch_size=batch_size,
        shuffle=shuffle
    )

def train_model(model, X, y, epochs=5, lr=1e-3, batch_size=512, verbose=False):
    loader = make_loader(X, y, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        total_n = 0

        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(xb)
            total_n += len(xb)

        if verbose:
            print(f"Epoch {epoch + 1:02d} | loss={total_loss / total_n:.5f}")

    return model

def predict_prob(model, X):
    model.eval()
    X_t = torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
        return torch.sigmoid(model(X_t)).cpu().numpy()

def evaluate_classifier(name, y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    return {
        "Model": name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, prob)
    }

def best_f1_threshold(y_true, prob):
    # Threshold selection uses validation data only.
    thresholds = np.linspace(0.05, 0.95, 181)
    scores = [
        f1_score(y_true, (prob >= t).astype(int), zero_division=0)
        for t in thresholds
    ]
    return float(thresholds[int(np.argmax(scores))])


## 4 — Run a real federated training round
**Live operation:** local training on both clients, sample-weighted averaging of compatible heads, and return of a shared head to both clients.

In [8]:
def weighted_average_predictors(predictors, sample_counts):
    total = float(sum(sample_counts))
    avg_state = copy.deepcopy(predictors[0].state_dict())

    for key in avg_state:
        avg_state[key] = torch.zeros_like(avg_state[key])
        for predictor, n in zip(predictors, sample_counts):
            avg_state[key] += predictor.state_dict()[key] * (n / total)

    global_predictor = SharedPredictor()
    global_predictor.load_state_dict(avg_state)
    return global_predictor

def federated_train(
    X_A, y_A, X_B, y_B,
    rounds=5, local_epochs=3, lr=1e-3, batch_size=512
):
    torch.manual_seed(SEED)

    encoder_A = ClientEncoder(X_A.shape[1])
    encoder_B = ClientEncoder(X_B.shape[1])
    global_predictor = SharedPredictor()

    history = []

    for rnd in range(1, rounds + 1):
        # Each client starts the round from the same global predictor.
        predictor_A = copy.deepcopy(global_predictor)
        predictor_B = copy.deepcopy(global_predictor)

        model_A = ClientModel(encoder_A, predictor_A)
        model_B = ClientModel(encoder_B, predictor_B)

        train_model(
            model_A, X_A, y_A,
            epochs=local_epochs, lr=lr,
            batch_size=batch_size, verbose=False
        )
        train_model(
            model_B, X_B, y_B,
            epochs=local_epochs, lr=lr,
            batch_size=batch_size, verbose=False
        )

        # Encoders have been updated locally through model_A/model_B.
        encoder_A = model_A.encoder
        encoder_B = model_B.encoder

        # Only compatible predictor parameters are federated.
        global_predictor = weighted_average_predictors(
            [model_A.predictor, model_B.predictor],
            [len(X_A), len(X_B)]
        )

        fed_A = ClientModel(encoder_A, copy.deepcopy(global_predictor))
        fed_B = ClientModel(encoder_B, copy.deepcopy(global_predictor))

        history.append({"Round": rnd})
        print(f"Completed federated round {rnd}/{rounds}")

    final_A = ClientModel(encoder_A, copy.deepcopy(global_predictor))
    final_B = ClientModel(encoder_B, copy.deepcopy(global_predictor))

    return final_A, final_B, global_predictor, history

fed_model_A, fed_model_B, global_predictor_demo, history = federated_train(
    A_train, y1_tr,
    B_train, y2_tr,
    rounds=3,
    local_epochs=3,
    batch_size=256
)
print("Shared head weights match:", all(torch.equal(fed_model_A.predictor.state_dict()[k], fed_model_B.predictor.state_dict()[k]) for k in global_predictor_demo.state_dict()))
print("Encoders remain distinct objects:", fed_model_A.encoder is not fed_model_B.encoder)

Completed federated round 1/3
Completed federated round 2/3
Completed federated round 3/3
Shared head weights match: True
Encoders remain distinct objects: True


## 5 — Show working predictions
These are **demonstration-only** results from a small training run.

In [9]:
# Select thresholds using validation data only
threshold_A = best_f1_threshold(
    y1_val, predict_prob(fed_model_A, A_val)
)
threshold_B = best_f1_threshold(
    y2_val, predict_prob(fed_model_B, B_val)
)

# Evaluate once on held-out test data
demo_results = pd.DataFrame([
    evaluate_classifier(
        "Client A — demo federated",
        y1_te,
        predict_prob(fed_model_A, A_test),
        threshold=threshold_A
    ),
    evaluate_classifier(
        "Client B — demo federated",
        y2_te,
        predict_prob(fed_model_B, B_test),
        threshold=threshold_B
    )
])

display(demo_results.round(3))

,Model,Threshold,Accuracy,Precision,Recall,F1,ROC-AUC
0,Client A — demo federated,0.430,0.652,0.171,0.804,0.282,0.779
1,Client B — demo federated,0.515,0.818,0.831,0.875,0.853,0.906


## 6 — Full research results: V2 and alignment approaches
These tables are **saved outputs of the original experiments**, not newly calculated by the shortened demo. V3 uses class-conditional prototypes, V4 uses contrastive alignment, and V5 uses class-conditional mean/covariance alignment.

In [10]:

# Display the original V2–V5 experimental summaries
for name, filename in [
    ("V2 — no alignment", "prototype_v2_multiseed_summary.csv"),
    ("V3 — prototype alignment", "prototype_v3_multiseed_summary.csv"),
    ("V4 — contrastive alignment", "prototype_contrastive_multiseed_summary.csv"),
    ("V5 — CORAL-based alignment", "prototype_coral_multiseed_summary.csv"),
]:
    print("\n", name)
    display(pd.read_csv(MODELS_DIR / filename).round(4))

# Additional dataset CSV has two header rows
print("\nAdditional dataset pair — Pima and Early Stage")

additional_results = pd.read_csv(
    MODELS_DIR / "additional_pair_multiseed_summary.csv",
    header=[0, 1]
)

# Flatten multi-level column names
additional_results.columns = [
    "Dataset" if i == 0 else
    "Approach" if i == 1 else
    f"{first} ({second})"
    for i, (first, second) in enumerate(additional_results.columns)
]

# Remove the extra header row included in the CSV
additional_results = additional_results[
    additional_results["Dataset"].isin(["Early Stage", "Pima"])
].copy()

# Convert metric columns to numbers
for col in additional_results.columns[2:]:
    additional_results[col] = pd.to_numeric(
        additional_results[col], errors="coerce"
    )

display(additional_results.reset_index(drop=True).round(4))


 V2 — no alignment


,Dataset,Approach,Accuracy,Precision,Recall,F1,ROC-AUC
0,Dataset 1,Federated,0.9683 ± 0.0037,0.9414 ± 0.0443,0.6700 ± 0.0164,0.7826 ± 0.0229,0.9691 ± 0.0036
1,Dataset 1,Local,0.9688 ± 0.0020,0.9379 ± 0.0212,0.6775 ± 0.0122,0.7867 ± 0.0131,0.9736 ± 0.0018
2,Dataset 2,Federated,0.9120 ± 0.0007,0.9892 ± 0.0029,0.8627 ± 0.0024,0.9216 ± 0.0006,0.9418 ± 0.0003
3,Dataset 2,Local,0.9112 ± 0.0017,0.9883 ± 0.0060,0.8623 ± 0.0035,0.9210 ± 0.0013,0.9417 ± 0.0003



 V3 — prototype alignment


,Dataset,Approach,Accuracy,Precision,Recall,F1,ROC-AUC
0,Dataset 1,Federated V3,0.9710 ± 0.0006,0.9671 ± 0.0167,0.6819 ± 0.0059,0.7997 ± 0.0020,0.9746 ± 0.0011
1,Dataset 1,Local,0.9688 ± 0.0020,0.9379 ± 0.0212,0.6775 ± 0.0122,0.7867 ± 0.0131,0.9736 ± 0.0018
2,Dataset 2,Federated V3,0.9132 ± 0.0015,0.9904 ± 0.0032,0.8638 ± 0.0011,0.9227 ± 0.0012,0.9420 ± 0.0004
3,Dataset 2,Local,0.9112 ± 0.0017,0.9883 ± 0.0060,0.8623 ± 0.0035,0.9210 ± 0.0013,0.9417 ± 0.0003



 V4 — contrastive alignment


,Dataset,Approach,Accuracy,Precision,Recall,F1,ROC-AUC
0,Dataset 1,Federated Contrastive,0.9626 ± 0.0040,0.8409 ± 0.0586,0.6965 ± 0.0291,0.7603 ± 0.0179,0.9668 ± 0.0034
1,Dataset 1,Local,0.9688 ± 0.0020,0.9379 ± 0.0212,0.6775 ± 0.0122,0.7867 ± 0.0131,0.9736 ± 0.0018
2,Dataset 2,Federated Contrastive,0.9088 ± 0.0005,0.9888 ± 0.0032,0.8577 ± 0.0022,0.9186 ± 0.0003,0.9406 ± 0.0006
3,Dataset 2,Local,0.9112 ± 0.0017,0.9883 ± 0.0060,0.8623 ± 0.0035,0.9210 ± 0.0013,0.9417 ± 0.0003



 V5 — CORAL-based alignment


,Dataset,Approach,Accuracy,Precision,Recall,F1,ROC-AUC
0,Dataset 1,Federated Coral,0.9709 ± 0.0009,0.9721 ± 0.0135,0.6767 ± 0.0068,0.7979 ± 0.0056,0.9747 ± 0.0006
1,Dataset 1,Local,0.9688 ± 0.0020,0.9379 ± 0.0212,0.6775 ± 0.0122,0.7867 ± 0.0131,0.9736 ± 0.0018
2,Dataset 2,Federated Coral,0.9133 ± 0.0014,0.9895 ± 0.0037,0.8647 ± 0.0017,0.9229 ± 0.0011,0.9419 ± 0.0006
3,Dataset 2,Local,0.9112 ± 0.0017,0.9883 ± 0.0060,0.8623 ± 0.0035,0.9210 ± 0.0013,0.9417 ± 0.0003



Additional dataset pair — Pima and Early Stage


,Dataset,Approach,Accuracy (mean),Accuracy (std),Precision (mean),Precision (std),...,F1 (mean),F1 (std),ROC-AUC (mean),ROC-AUC (std),Threshold (mean),Threshold (std)
0,Early Stage,Federated V3,0.7308,0.0625,0.7823,0.0566,...,0.8050,0.0588,0.8307,0.0854,0.457,0.0670
1,Early Stage,Federated V5,0.7402,0.0566,0.7848,0.0606,...,0.8147,0.0460,0.8295,0.0867,0.457,0.0670
2,Early Stage,Local,0.7645,0.0483,0.8707,0.0770,...,0.8126,0.0563,0.8709,0.0437,0.503,0.0566
3,Pima,Federated V3,0.6403,0.1373,0.5210,0.1124,...,0.5569,0.0828,0.7127,0.1311,0.456,0.0671
4,Pima,Federated V5,0.6403,0.1373,0.5210,0.1124,...,0.5569,0.0828,0.7123,0.1314,0.456,0.0671
5,Pima,Local,0.6896,0.0354,0.5480,0.0376,...,0.5833,0.0687,0.7443,0.0379,0.454,0.0531
